# Sky model and bandpass templates

Two kinds of signal are not described by one map shared by all the detectors,
and bias the calibration (see the calibration notebook):

- **structure below the pixel size**: the samples see the sky at their exact
  pointing, the map has one value per pixel. Given as alm, a `sky_model` is
  interpolated at the pointing (`ducc0.sht.synthesis_general`) and subtracted;
  its binned map is put back, so the maps are still those of the whole sky.
- **bandpass mismatch**: detectors of the same channel see the foregrounds
  through slightly different bands. A sky template, `Template(sky=...,
  sum_zero=True)`, gets one amplitude per detector (summing to zero: the
  common part is the map), fitted with the maps
  and the gains. With the derivative of the foreground with respect to
  frequency, $F_i \simeq F + \Delta\nu_i\,\partial F/\partial\nu$, the
  amplitude is the shift of the band $\Delta\nu_i$ (the foreground in the sky
  model and its derivative at the same reference frequency).

Here: the data of the calibration notebook (4 detectors, 1/f noise, gains with
1% offsets and a 3% drift), with a CMB and a dust foreground at 140 GHz: 3 mK
in a Galactic band with structure at 1°, 10% polarized, spectral index
$\beta = 1.5 \pm 0.2$ varying across the sky, seen by each detector through its
band, shifted by $\Delta\nu_i$ = +1.5, −1.0, +0.8, −1.3 GHz. The sky is
simulated at nside 128, the maps are at nside 16.

In [1]:
import dataclasses
import ducc0
import healpy as hp
import numpy as np
from pysanepic import CalibrationParameters, DetectorData, GLSParameters, OneOverFNoise, Pointing, Template, make_maps

nside, fs, n = 16, 10.0, 240_000
hi, lmax = 128, 256
rng = np.random.default_rng(3)
npix, nhi = 12 * nside**2, 12 * hi**2
z, zh = hp.pix2vec(nside, np.arange(npix))[2], hp.pix2vec(hi, np.arange(nhi))[2]
cmb_alm = hp.map2alm(hp.smoothing(rng.normal(0, 1e-4, (3, nhi)), fwhm=np.radians(6)), lmax=lmax)
# dust: amplitude A (3 mK in a band, structure at 1 degree), 10% polarized, spectral index beta(p)
A = np.where(np.abs(zh) < 0.2, 3e-3, 2e-5) * (1 + 0.3 * hp.smoothing(rng.normal(0, 3, nhi), fwhm=np.radians(1)))
A = np.maximum(A, 0)
pa = hp.smoothing(rng.normal(0, 1, nhi), fwhm=np.radians(3)) * np.pi
dust = np.array([A, 0.1 * A * np.cos(2 * pa), 0.1 * A * np.sin(2 * pa)])
beta = 1.5 + 0.2 * hp.smoothing(rng.normal(0, 3, nhi), fwhm=np.radians(5))
nu0 = 140.0
shift = np.array([1.5, -1.0, 0.8, -1.3])  # GHz: band centre of each detector minus nu0
alm = lambda m: hp.map2alm(m, lmax=lmax)
F, F1 = alm(dust), alm(dust * beta / nu0)  # the dust at nu0 and its derivative dF/dnu
noise = OneOverFNoise(net_ukrts=35.0, fknee_hz=0.05, alpha=1.0, fmin_hz=1e-4)
t = np.arange(n)
lam = 2 * np.pi * t / n + rng.uniform(0, 2 * np.pi)
spin = 2 * np.pi * t / 3000.0
a = np.array([np.cos(lam), np.sin(lam), np.zeros(n)])
e2 = np.cross(a.T, [0, 0, 1]).T
vec = np.cos(1.3) * a + np.sin(1.3) * (np.cos(spin) * np.array([[0], [0], [1]]) + np.sin(spin) * e2)
theta, phi = np.arccos(np.clip(vec[2], -1, 1)), np.arctan2(vec[1], vec[0]) % (2 * np.pi)
psi = (spin + lam) % (2 * np.pi) - np.pi
solar = 3.36e-3 * np.array([0.3, -0.5, 0.8]) / np.linalg.norm([0.3, -0.5, 0.8])
dipole = solar @ vec + np.sum(2.7e-4 * np.array([-np.sin(lam), np.cos(lam), 0 * t]) * vec, axis=0)
loc = np.stack([theta, phi], axis=1)


def interp(alm, ang):
    i = ducc0.sht.synthesis_general(alm=alm[:1], spin=0, lmax=lmax, loc=loc, epsilon=1e-10)[0]
    q, u = ducc0.sht.synthesis_general(alm=alm[1:], spin=2, lmax=lmax, loc=loc, epsilon=1e-10)
    return i + q * np.cos(2 * ang) + u * np.sin(2 * ang)


def draw(n):
    return np.fft.irfft(np.fft.rfft(rng.normal(size=n)) / np.sqrt(noise.weights(n, fs)), n=n)


data, true_gain = [], []
for i, pol_angle in enumerate(np.radians([0, 90, 45, 135])):
    ang = psi + pol_angle
    dust_i = alm(dust * ((nu0 + shift[i]) / nu0) ** beta)  # the dust through the band of detector i
    signal = interp(cmb_alm, ang) + interp(dust_i, ang) + dipole
    g = 1 + 0.01 * rng.normal() + 0.03 * np.cumsum(rng.normal(size=n)) / np.sqrt(n)
    true_gain.append(g)
    data.append(DetectorData(tod=g * signal + draw(n), pointing=Pointing(theta=theta, phi=phi, psi=psi, coordinates="E"),
                             sampling_rate_hz=fs, noise=noise, pol_angle_rad=pol_angle, dipole=dipole, name=f"d{i}"))
params = GLSParameters(chunk_s=2400.0, tol=1e-12)
galaxy = (np.abs(z) > 0.3).astype(float)  # a mask: 1 = used, 0 on the Galactic band

The reference: maps from the data calibrated with the true gains, with the
exact model and the bandpass fitted. The rest is compared to it outside the
Galactic band, in µK; gains as in the calibration notebook (rms of
$g/g_\text{true} - 1$ and its mean, the absolute calibration).

In [2]:
def report(name, r):
    ok = (r.maps[1] != hp.UNSEEN) & (np.abs(z) > 0.3)
    e = (r.maps - ref.maps)[:, ok] * 1e6
    gerr = np.array([g[t // 12000] / g0 - 1 for g, g0 in zip(r.gains, true_gain)])
    line = f"{name:30s} I/Q/U {e[0].std():5.2f} / {e[1].std():5.2f} / {e[2].std():5.2f} uK   gains rms {np.sqrt(np.mean(gerr**2)):.1e}, absolute {gerr.mean():+.1e}"
    for k, b in (r.templates or {}).items():
        line += f"\n{'':30s} {k}: " + ", ".join(f"{v:+.3g}" for v in b.values())
    print(line)


model = cmb_alm + F  # the sky at nu0
calibrated = [dataclasses.replace(d, tod=d.tod / g - dipole, dipole=None) for d, g in zip(data, true_gain)]
ref = make_maps(calibrated, nside, coordinates="E", params=params, sky_model=model, templates={"fg_deriv1": Template(sky=F1, sum_zero=True)})
cal = CalibrationParameters(gain_period_s=1200.0)
print("band shifts - their mean [GHz]:", shift - shift.mean())

band shifts - their mean [GHz]: [ 1.5 -1.   0.8 -1.3]


## Structure below the pixel size

Without a model the Galaxy has to be masked for the calibration, and its
structure below the pixel size still leaks into the I map through the 1/f
weights. With the model as alm, no mask is needed and the I map is right; Q
and U are still off, as the gains are biased by the bandpass mismatch.

In [3]:
report("no model, Galactic mask", make_maps(data, nside, coordinates="E", params=params,
                                            calibration=CalibrationParameters(gain_period_s=1200.0, mask=galaxy)))
report("sky model (alm), no mask", make_maps(data, nside, coordinates="E", params=params, calibration=cal, sky_model=model))

no model, Galactic mask        I/Q/U 19.27 /  5.16 /  5.06 uK   gains rms 4.7e-03, absolute -3.1e-03


sky model (alm), no mask       I/Q/U  0.93 /  3.80 /  3.68 uK   gains rms 6.3e-03, absolute +1.6e-04


## Bandpass mismatch

The dust seen by each detector differs by a few percent: without a template
the difference goes into the gains and, between the two detectors of a pair,
into Q and U. A template per detector removes it: the foreground itself
(`"fg"`, one scale per detector) or its first derivative (`"fg_deriv1"`, whose
amplitudes are the band shifts in GHz).

In [4]:
for name, templates in (("bandpass: fg", {"fg": Template(sky=F, sum_zero=True)}),
                        ("bandpass: fg_deriv1", {"fg_deriv1": Template(sky=F1, sum_zero=True)})):
    report(name, make_maps(data, nside, coordinates="E", params=params, calibration=cal,
                           sky_model=model, templates=templates))

bandpass: fg                   I/Q/U  0.93 /  1.26 /  1.31 uK   gains rms 3.3e-03, absolute +1.9e-04
                               fg: +0.0155, -0.00911, +0.00817, -0.0145


bandpass: fg_deriv1            I/Q/U  0.93 /  1.26 /  1.31 uK   gains rms 3.3e-03, absolute +1.9e-04
                               fg_deriv1: +1.44, -0.847, +0.757, -1.35


The first derivative recovers the band shifts within the noise, and Q/U
drop from 3.8 to 1.3 µK; `"fg"` alone does as well here, as the spectral
index varies little. Higher orders of the expansion can be added as more
templates if needed (see the documentation).

## Masks

Each fit has its own mask: `CalibrationParameters(mask=...)` for the gains,
`Template(mask=...)` for each template, e.g. `galaxy * sources` for the gains
and `sources` for the bandpass template: its amplitudes are fitted on the
Galaxy, where the foreground is bright, and point sources missing from the
model are left out of everything. A template without a mask has that of
the gains. Samples left out of the gain fit only are calibrated with the
gains of the previous iteration, so the iterations converge slower
(`make_maps` warns); with a good model, as here, no mask is needed.

## Without calibration

Both corrections are useful without calibration too. Here the data are
calibrated with the true gains and solved without `calibration`: the
structure below the pixel size leaks through the 1/f weights all over the
map (pixelization noise, also far from the Galactic band), and the bandpass
mismatch leaks I into Q and U. Differences from the reference (the same
noise, so they are the systematic errors only), out of and on the band:

In [5]:
def compare(name, r):
    line = f"{name:30s}"
    for sel in (np.abs(z) > 0.3, np.abs(z) < 0.3):
        ok = (r.maps[1] != hp.UNSEEN) & sel
        e = (r.maps - ref.maps)[:, ok] * 1e6
        line += f" | I/Q/U {e[0].std():6.2f} / {e[1].std():6.2f} / {e[2].std():6.2f} uK"
    print(line)


for name, kw in (("nothing", {}), ("sky model as maps", dict(sky_model=hp.alm2map(model, nside, lmax=lmax))),
                 ("sky model as alm", dict(sky_model=model)),
                 ("sky model + fg_deriv1", dict(sky_model=model, templates={"fg_deriv1": Template(sky=F1, sum_zero=True)}))):
    compare(name, make_maps(calibrated, nside, coordinates="E", params=params, **kw))

nothing                        | I/Q/U  17.96 /   4.84 /   4.88 uK | I/Q/U  99.25 /  25.40 /  31.08 uK


sky model as maps              | I/Q/U  17.96 /   4.84 /   4.88 uK | I/Q/U  99.25 /  25.40 /  31.08 uK


sky model as alm               | I/Q/U   0.03 /   3.31 /   3.37 uK | I/Q/U   1.04 /  21.50 /  21.62 uK


sky model + fg_deriv1          | I/Q/U   0.00 /   0.00 /   0.00 uK | I/Q/U   0.00 /   0.00 /   0.00 uK


The model as maps changes nothing (Q/U are solved in every pixel here); as
alm it removes the leakage of the structure below the pixel size from I; the
bandpass template removes what is left in Q/U (the last line is the
reference itself). The case is extreme on purpose (pixels of 3.7°, structure
at 1°): at nside 512 the pixelization noise is much smaller, but it is there
where the gradients are steep, on the Galactic plane and around bright point
sources.

## Covariance of the amplitudes

With the second order of the expansion too, $\tfrac{1}{2}\,\partial^2F/\partial\nu^2$
(amplitude $\Delta\nu_i^2$, minus its mean), the two templates of a detector
are nearly proportional when the spectral index varies little:
$\partial^2F/\partial\nu^2 \simeq (\beta - 1)/\nu\;\partial F/\partial\nu$.
`template_covariance=True` gives the covariance of the amplitudes,
marginalized over the maps (one more PCG solve per amplitude, 8 here). The
data are calibrated with the true gains, without the dust model at the band
of each detector, so the expansion is the only model of the mismatch.

In [6]:
F2 = alm(dust * beta * (beta - 1) / (2 * nu0**2))  # 1/2 d2F/dnu2: amplitude = shift^2
kw = dict(coordinates="E", params=params, sky_model=model, template_covariance=True)
two = make_maps(calibrated, nside, templates={"fg_deriv1": Template(sky=F1, sum_zero=True),
                                              "fg_deriv2": Template(sky=F2, sum_zero=True)}, **kw)
one = make_maps(calibrated, nside, templates={"fg_deriv1": Template(sky=F1, sum_zero=True)}, **kw)
expect = {"fg_deriv1": shift - shift.mean(), "fg_deriv2": shift**2 - np.mean(shift**2)}
for name, r in (("fg_deriv1 alone", one), ("fg_deriv1 + fg_deriv2", two)):
    C = r.template_covariance["matrix"]
    sigma = np.sqrt(np.diag(C))
    print(name)
    for (k, det), s in zip(r.template_covariance["labels"], sigma):
        print(f"  {k} {det}: fit {r.templates[k][det]:+7.3f} +- {s:.3f}, true {expect[k][int(det[1:])]:+7.3f}")
corr = two.template_covariance["matrix"] / np.outer(*2 * [np.sqrt(np.diag(two.template_covariance["matrix"]))])
print("correlation of fg_deriv1 and fg_deriv2 of each detector:", np.round(np.diag(corr[:4, 4:]), 3))

fg_deriv1 alone
  fg_deriv1 d0: fit  +1.409 +- 0.041, true  +1.500
  fg_deriv1 d1: fit  -0.910 +- 0.041, true  -1.000
  fg_deriv1 d2: fit  +0.818 +- 0.041, true  +0.800
  fg_deriv1 d3: fit  -1.317 +- 0.041, true  -1.300
fg_deriv1 + fg_deriv2
  fg_deriv1 d0: fit  +0.734 +- 0.409, true  +1.500
  fg_deriv1 d1: fit  -1.156 +- 0.410, true  -1.000
  fg_deriv1 d2: fit  +1.268 +- 0.408, true  +0.800
  fg_deriv1 d3: fit  -0.847 +- 0.411, true  -1.300
  fg_deriv2 d0: fit +373.735 +- 225.484, true  +0.855
  fg_deriv2 d1: fit +136.736 +- 225.926, true  -0.395
  fg_deriv2 d2: fit -249.897 +- 224.651, true  -0.755
  fg_deriv2 d3: fit -260.574 +- 226.681, true  +0.295
correlation of fg_deriv1 and fg_deriv2 of each detector: [-0.995 -0.995 -0.995 -0.995]


Alone, the first derivative measures the shifts to ±0.04 GHz (within ~2σ of
the truth: the second order it neglects is small here). With the second
order the two amplitudes of each detector are 99.5% anticorrelated: the
error on the shifts grows tenfold and the second-order amplitudes are not
measured at all (±225 GHz² for true values below 1 GHz²), while the maps do
not change. Fitted values far from the truth with such errors are the
degeneracy, not a bug. Add higher orders only where the data constrain them
(a bright foreground and low noise, as on the Galactic plane of a long run),
and check their covariance.